# TopicGPT: LLM-based Topic Modeling

Implementation of TopicGPT (NAACL 2024) with modifications:
- Uses **local LM Studio API** (`mistralai/ministral-3-3b`)
- 3-stage pipeline: **Generation → Refinement → Assignment**
- Evaluates with **Coherence (C_v)**, **IRBO Diversity**, and **Topic Quality**
- Checkpointing support for resumable execution

In [1]:
import os
import re
import json
import time
import random
import pickle
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

In [2]:
# ─── Subjects & Paths ───
LIST_SUBJECT = ["cs", "math", "physics"]
VERSION = "v1"

BASE_DIR = Path("../../../../data/preprocess")
RESULT_DIR = Path("../../../../results/topicGpt/modeling")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)
    (CHECKPOINT_DIR / subject).mkdir(parents=True, exist_ok=True)

# ─── LM Studio API Config ───
LLM_API_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "mistralai/ministral-3-3b"
LLM_TEMPERATURE = 0.2
LLM_MAX_TOKENS = 39000

# ─── TopicGPT Pipeline Config ───
GENERATION_SAMPLE_SIZE = 500      # docs sampled per generation batch
GENERATION_BATCH_SIZE = 5         # docs per LLM call in generation
GENERATION_MAX_BATCHES = 100      # max LLM calls for generation
ASSIGNMENT_BATCH_SIZE = 5       # docs per LLM call in assignment

# ─── Coherence Config ───
TOP_N_WORDS = 10
RBO_P = 0.9

print(f"Subjects: {LIST_SUBJECT}")
print(f"LLM: {LLM_MODEL} @ {LLM_API_URL}")
print(f"Results: {RESULT_DIR.resolve()}")

Subjects: ['cs', 'math', 'physics']
LLM: mistralai/ministral-3-3b @ http://localhost:1234/v1/chat/completions
Results: /home/nedo/Kuliah/TA/Program/results/topicGpt/modeling


## LLM API Helper

In [3]:
def call_llm(system_prompt: str, user_prompt: str, max_retries: int = 3) -> str:
    """Call LM Studio API with retry logic."""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS,
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                LLM_API_URL,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            
            # Handle both OpenAI-style and LM Studio response formats
            if "choices" in data:
                return data["choices"][0]["message"]["content"].strip()
            elif "content" in data:
                return data["content"].strip()
            elif "output" in data:
                return data["output"].strip()
            else:
                return str(data)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  LLM call failed after {max_retries} attempts: {e}")
                return ""

# Test connection
test_resp = call_llm("You are a helpful assistant.", "Say 'OK' if you can read this.")
print(f"LLM connection test: {test_resp[:100]}")

LLM connection test: OK! How can I assist you today?


## Checkpoint Utilities

In [4]:
def save_checkpoint(data, name: str, subject: str):
    """Save checkpoint to disk."""
    path = CHECKPOINT_DIR / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = CHECKPOINT_DIR / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## Data Loading

In [5]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load preprocessed dataset."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    return df

all_data = {}
for subject in LIST_SUBJECT:
    df = load_dataset(subject)
    all_data[subject] = df
    print(f"{subject}: {len(df):,} documents loaded")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Sample text: {str(df['text'].iloc[0])[:120]}...")

cs: 165,756 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: fault detection using immune based systems and formal language algorithms this paper describes two approaches for fault ...
math: 157,085 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: supersymmetry and homotopy the homotopical information hidden in a supersymmetric structure is revealed by considering d...
physics: 146,311 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: critical dynamics of gelation shear relaxation and dynamic density fluctuations are studied within a rouse model general...


---
## Stage 1: Topic Generation

Iteratively prompt the LLM with batches of documents. For each batch, the LLM sees existing topics and either identifies existing topics or proposes new ones.

In [6]:
GENERATION_SYSTEM_PROMPT = """You are an expert topic analyst. Your task is to identify generalizable topics from academic paper abstracts. 

Rules:
- Each topic must be GENERAL and BROAD enough to cover multiple papers
- Topic labels must be concise (2-5 words)
- Each topic needs a short description
- Do NOT create overly specific topics tied to a single paper
- Each topic should represent a SINGLE concept, not a combination
- Return ONLY the topic lines, nothing else"""

GENERATION_USER_TEMPLATE = """Below are the current topics discovered so far:

[Current Topics]
{topics}

[Documents]
{documents}

[Instructions]
1. Read each document above.
2. For each document, determine which existing topic it belongs to, OR propose a new topic if none fit.
3. Only add a new topic if it is truly distinct from all existing topics.
4. Use the format: [ID] Topic Label: Short description

Return ONLY the NEW topics (if any) in the format:
[NEW] Topic Label: Short description

If all documents fit existing topics, return:
None

Your response:"""

In [7]:
def format_topics_for_prompt(topics: dict) -> str:
    """Format topics for prompt."""
    if not topics:
        return "None"
    lines = []
    for tid, t in sorted(topics.items()):
        lines.append(f"[{tid}] {t['label']}: {t['description']}")
    return "\n".join(lines)

def parse_generated_topics(response: str, topics: dict) -> list:
    """Parse the newly generated topics from the LLM response."""
    new_topics = []
    if not response or response.strip().lower() == "none":
        return new_topics
    
    import re
    # Split by actual newline characters in the response text
    for line in response.strip().split("\n"):
        line = line.strip()
        match = re.match(r'\[NEW\]\s+(.+?):\s*(.+)', line, re.IGNORECASE)
        if match:
            new_topics.append({
                "label": match.group(1).strip(),
                "description": match.group(2).strip()
            })
    return new_topics

def stratified_sample_by_year(df: pd.DataFrame, min_per_year: int = 100, pct: float = 0.05) -> list:
    """Stratified sampling: min(min_per_year, 5%) docs per year for fair temporal representation."""
    random.seed(42)
    df = df.copy()
    df['year'] = pd.to_datetime(df['submitted_date'], errors='coerce').dt.year
    sampled_indices = []
    for year, group in df.groupby('year'):
        n = len(group)
        sample_n = max(min_per_year, int(n * pct))
        sample_n = min(sample_n, n)  # can't sample more than available
        sampled = group.sample(n=sample_n, random_state=42)
        sampled_indices.extend(sampled.index.tolist())
    random.shuffle(sampled_indices)
    print(f"  Stratified sample: {len(sampled_indices)} docs from {df['year'].nunique()} years")
    return sampled_indices


def generate_topics(df: pd.DataFrame, subject: str) -> dict:
    """Stage 1: Iterative topic generation from document batches."""
    
    # Try loading checkpoint
    checkpoint = load_checkpoint("generation", subject)
    if checkpoint is not None:
        topics = checkpoint["topics"]
        processed = checkpoint["processed"]
        print(f"  Resumed: {len(topics)} topics, {processed} batches processed")
    else:
        topics = {}  # {id: {label, description}}
        processed = 0
    
    # Stratified per-year sampling: min 100 docs or 5% per year
    texts = df['text'].fillna('').tolist()
    sampled_indices = stratified_sample_by_year(df, min_per_year=100, pct=0.05)
    sampled_texts = [texts[i] for i in sampled_indices]
    # sampled_texts = texts  # use all texts directly
    # sampled_texts = texts  # use all texts directly
    
    # Process in batches
    # Process all texts in batches
    n_batches = len(sampled_texts) // GENERATION_BATCH_SIZE
    
    next_id = max(topics.keys(), default=0) + 1
    stable_count = 0  # consecutive batches with no new topics
    
    pbar = tqdm(range(processed, n_batches), desc="Generating topics", initial=processed, total=n_batches)
    if pbar.n >= pbar.total:
        return topics
    
    for batch_idx in pbar:
        start = batch_idx * GENERATION_BATCH_SIZE
        end = start + GENERATION_BATCH_SIZE
        batch_texts = sampled_texts[start:end]
        
        # Format documents
        docs_str = "\n\n".join([f"Document {i+1}:\n{t[:500]}" for i, t in enumerate(batch_texts)])
        topics_str = format_topics_for_prompt(topics)
        
        user_prompt = GENERATION_USER_TEMPLATE.format(
            topics=topics_str,
            documents=docs_str
        )
        
        response = call_llm(GENERATION_SYSTEM_PROMPT, user_prompt)
        new_topics = parse_generated_topics(response, topics)
        
        if new_topics:
            stable_count = 0
            for nt in new_topics:
                topics[next_id] = nt
                next_id += 1
        else:
            stable_count += 1
        
        pbar.set_postfix({'topics': len(topics), 'stable_count': stable_count})
        
        # Checkpoint every 10 batches
        if (batch_idx + 1) % 10 == 0:
            save_checkpoint({"topics": topics, "processed": batch_idx + 1}, "generation", subject)
        
        # Early stop if stable for 50 consecutive batches
        if stable_count >= 50:
            print(f"  Early stop condition met after {stable_count} batches, but continuing to process all texts...")
            # break
    
    # Final save
    save_checkpoint({"topics": topics, "processed": batch_idx + 1}, "generation", subject)
    print(f"  Generated {len(topics)} topics")
    return topics


In [8]:
# Run Topic Generation for all subjects
all_topics = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC GENERATION: {subject.upper()}")
    print(f"{'='*60}")
    
    topics = generate_topics(all_data[subject], subject)
    all_topics[subject] = topics
    
    print(f"\n  Topics for {subject}:")
    for tid, t in sorted(topics.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")


TOPIC GENERATION: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/generation.pkl
  Resumed: 276 topics, 1781 batches processed
  Stratified sample: 8906 docs from 26 years


Generating topics: 100%|██████████| 1781/1781 [00:00<?, ?it/s]



  Topics for cs:
    [1] **Graph Neural Networks: ** Advanced models for heterogeneous data representation and relational reasoning.
    [2] **Adversarial Robustness: ** Techniques to defend machine learning systems against input perturbations and attacks.
    [3] **Neural Architecture Pruning: ** Efficient model compression via parameter reduction and fine-tuning optimization.
    [4] **Cooperative Learning Systems: ** Multi-agent information exchange for unified perception tasks.
    [5] **Knowledge Representation Corpora: ** Large-scale annotated datasets linking concepts to structured knowledge bases and textual definitions.
    [6] **Single-Document Summarization: ** Techniques for compressing and constraining text extraction in unstructured single-document settings.
    [7] **De Novo Assembly Algorithms: ** Methods for constructing lossless string representations from sequencing reads.
    [8] **Wireless Channel Optimization: ** Analytic approaches to maximize throughput in corr

Generating topics: 100%|██████████| 1609/1609 [00:00<?, ?it/s]


  Topics for math:
    [1] **Algebraic Curve Parametrization**: Radical methods for parametrizing irreducible curves over algebraically closed fields.
    [2] **Hardy Space Kernels**: Analytic properties of paired kernels in Lebesgue-Hilbert spaces on the unit circle.
    [3] **Infinite Horizon Optimization**: Optimal solutions to unbounded continuous-time infinite-horizon problems under overtaking criteria.
    [4] **Functional Analysis Algebras**: Weighted convolution properties in function spaces and algebras.
    [5] **Nonlinear PDE Coupling**: Strongly coupled systems with vanishing potentials in partial differential equations.
    [6] **Stochastic Functional Differential Equations**: Controllability analysis of delay/integral-driven systems with stochastic perturbations.
    [7] **Rational Homogeneous Submanifolds**: Geometric properties and classification of splitting submanifolds in complex rational homogeneous spaces.
    [8] **Nonconvex Harmonic Maps**: Characterization of i


Generating topics: 100%|██████████| 1513/1513 [00:00<?, ?it/s]


  Topics for physics:
    [1] **Fluid Dynamics Interactions**: Study of jet deflection and wake effects in coupled flapping systems, including propulsion mechanics and structural interactions.
    [2] **Evaporation-Driven Phenomena**: Analysis of early-stage droplet evaporation dynamics (e.g., coffee ring formation) under varying capillary/solutal conditions.
    [3] **Quantum Van der Waals Interactions**: Investigation of dipolar quantum fluctuations in van der waals complexes, focusing on mutual Coulomb effects at molecular scales.
    [4] **Quantum Readout Systems**: Advancements in high-fidelity nuclear/spin qubit detection via photonic interfaces and solid-state architectures.
    [5] **Reconfigurable Optoelectronic Amplifiers**: Study of temporal Bragg gratings as dynamic broadband parametric amplifiers with spatial/temporal modulation effects.
    [6] **Seismic Data Processing**: Blind deconvolution techniques for multichannel seismic data via spectral optimization and sparse r

---
## Stage 2: Topic Refinement

Merge near-duplicate or overlapping topics using the LLM.

In [9]:
REFINEMENT_SYSTEM_PROMPT = """You are an expert at organizing topic taxonomies. Your task is to merge topics that are near-duplicates, synonyms, or heavily overlapping.

Rules:
- Only merge topics that are truly redundant or nearly identical
- Keep the most general and descriptive label
- Return the merge operations in the specified format
- If no merges are needed, return "None" """

REFINEMENT_USER_TEMPLATE = """
You are given a list of topics. Identify groups that should be merged because they are near-duplicates or heavily overlapping.

[Topic List]
{topics}

[Task]
Detect topics representing the same concept and propose merges.

[Output Format]
Return ONLY valid JSON.

{{
  "merges": [
    {{
      "merge_ids": [id1, id2, ...],
      "kept_id": id,
      "label": "New merged topic label",
      "description": "Short description of the merged topic",
      "confidence": 0.0
    }}
  ]
}}

Rules:
- "merge_ids" must contain all topic IDs being merged
- "kept_id" must be one of the IDs inside merge_ids
- "confidence" must be between 0.0 and 1.0

If no merges are needed return:

{{
  "merges": []
}}

Return JSON only. Do not include explanations.
"""

In [10]:
import json

def parse_refinement_response(response: str, topics: dict, min_confidence: float = 0.0) -> list:
    """Parse merge operations from structured LLM JSON response."""
    
    if not response:
        return []

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        # fallback: extract JSON if wrapped in text
        start = response.find("{")
        end = response.rfind("}") + 1
        if start == -1 or end == -1:
            return []
        try:
            data = json.loads(response[start:end])
        except json.JSONDecodeError:
            return []

    merges = []

    for item in data.get("merges", []):
        merge_ids = item.get("merge_ids", [])
        kept_id = item.get("kept_id")
        confidence = float(item.get("confidence", 0.0))

        # validation
        if not merge_ids or kept_id not in merge_ids:
            continue

        if confidence < min_confidence:
            continue

        merges.append({
            "merge_ids": [int(x) for x in merge_ids],
            "kept_id": int(kept_id),
            "label": item.get("label", "").strip(),
            "description": item.get("description", "").strip(),
            "confidence": confidence
        })

    return merges


def refine_topics(topics: dict, subject: str, min_confidence: float = 0.7) -> dict:
    """Stage 2: Merge near-duplicate topics."""

    checkpoint = load_checkpoint("refinement", subject)
    if checkpoint is not None:
        return checkpoint

    refined = dict(topics)

    topics_str = format_topics_for_prompt(refined)
    user_prompt = REFINEMENT_USER_TEMPLATE.format(topics=topics_str)

    response = call_llm(REFINEMENT_SYSTEM_PROMPT, user_prompt)

    merges = parse_refinement_response(response, refined, min_confidence=min_confidence)

    if not merges:
        print("  No merges needed")
    else:
        # Apply strongest merges first
        merges = sorted(merges, key=lambda x: x["confidence"], reverse=True)

        used_ids = set()

        print(f"  Applying {len(merges)} merge(s):")

        for m in merges:

            merge_ids = set(m["merge_ids"])
            kept_id = m["kept_id"]

            # Skip if topics already merged
            if merge_ids & used_ids:
                continue

            # Validate existence
            valid_ids = [i for i in merge_ids if i in refined]
            if len(valid_ids) < 2:
                continue

            print(
                f"    Merge {valid_ids} -> [{kept_id}] "
                f"{m['label']} (conf={m['confidence']:.2f})"
            )

            # Update kept topic
            if kept_id in refined:
                refined[kept_id] = {
                    "label": m["label"],
                    "description": m["description"]
                }

            # Remove merged topics
            for mid in valid_ids:
                if mid != kept_id and mid in refined:
                    del refined[mid]

            used_ids.update(valid_ids)

    # ---- Reindex topics sequentially ----

    reindexed = {}
    old_to_new = {}

    for new_id, (old_id, topic) in enumerate(sorted(refined.items()), start=1):
        reindexed[new_id] = topic
        old_to_new[old_id] = new_id

    save_checkpoint(reindexed, "refinement", subject)

    print(f"  Refined: {len(topics)} -> {len(reindexed)} topics")

    return reindexed

In [11]:
# Run Topic Refinement
all_refined_topics = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC REFINEMENT: {subject.upper()}")
    print(f"{'='*60}")
    
    refined = refine_topics(all_topics[subject], subject)
    all_refined_topics[subject] = refined
    
    print(f"\n  Refined topics for {subject}:")
    for tid, t in sorted(refined.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")


TOPIC REFINEMENT: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/refinement.pkl

  Refined topics for cs:
    [1] Graph Neural Networks and Optimization Algorithms: Advanced models for heterogeneous data representation and relational reasoning in GNNs, including optimization algorithms for dynamic problem-solving.
    [2] Adversarial Robustness and Training: Techniques for defending ML systems against input perturbations and adversarial attacks, including optimization of gradient-based adversarial training methods.
    [3] Neural Architecture Pruning and Optimization: Efficient model compression via parameter reduction (pruning) and optimization techniques for balancing computational cost with performance.
    [4] **Cooperative Learning Systems: ** Multi-agent information exchange for unified perception tasks.
    [5] **Knowledge Representation Corpora: ** Large-scale annotated datasets linking concepts to structured knowledge bases and textual definitions.
    [6] **Single-Do

---
## Stage 3: Topic Assignment

Assign each document to the most relevant topic(s) and extract representative keywords.

In [12]:
ASSIGNMENT_SYSTEM_PROMPT = """You are an expert at categorizing academic papers into topics. Assign each document to the most relevant topic from the provided list.

Rules:
- You MUST use topics from the provided list. Do NOT invent new topics.
- Assign exactly ONE primary topic per document.
- Also extract 5-10 representative keywords from the document that characterize the topic.
- Keywords should be single words or bigrams, separated by commas.
- Return ONLY the formatted assignments, nothing else."""

ASSIGNMENT_USER_TEMPLATE = """
You are given a list of topics and a list of documents.

[Available Topics]
{topics}

[Documents]
{documents}

Task:
Assign the most relevant topic to each document.

Output ONLY valid JSON with the following structure:

{{
  "assignments": [
    {{
      "doc_id": 1,
      "topic_id": 3,
      "topic_label": "Topic label",
      "keywords": ["keyword1", "keyword2"],
      "confidence": 0.0
    }}
  ]
}}

Rules:
- doc_id starts from 1
- topic_id must be one of the available topic IDs
- confidence must be between 0.0 and 1.0
- keywords must be short lowercase tokens
- Every document MUST appear exactly once
- EVERY assignment object MUST contain ALL fields:
  doc_id, topic_id, topic_label, keywords, confidence
- Do not omit any field
- If unknown use:
  keywords: []
  confidence: 0.0
- Return ONLY JSON
"""

In [13]:

# def parse_llm_json(response: str):

#     if not response:
#         return None

#     # remove markdown fences
#     text = re.sub(r"```json|```", "", response).strip()

#     # extract main json block
#     start = text.find("{")
#     end = text.rfind("}") + 1
#     if start != -1 and end != -1:
#         text = text[start:end]

#     # fix missing brackets
#     open_braces = text.count("{")
#     close_braces = text.count("}")
#     if close_braces < open_braces:
#         text += "}" * (open_braces - close_braces)

#     open_brackets = text.count("[")
#     close_brackets = text.count("]")
#     if close_brackets < open_brackets:
#         text += "]" * (open_brackets - close_brackets)

#     try:
#         return json5.loads(text)
#     except Exception:
#         return None
# import json5
# import re

# def extract_assignments(text):
#     text = re.sub(r"```json|```", "", text)

#     m = re.search(r"\{[\s\S]*\}", text)
#     if not m:
#         return []
    
#     json_str = m.group(0)
    
#     json_str = re.sub(r'"([a-zA-Z_]+):', r'"\1":', json_str)
    
#     json_str = re.sub(r',\s*}', '}', json_str)
#     json_str = re.sub(r',\s*]', ']', json_str)
    
#     json_str = json_str.replace('\xa0', ' ')

#     try:
#         data = json5.loads(json_str)
#         return data.get("assignments", [])
#     except Exception as e:
#         return []
# import re
# import json5

# def extract_assignments(text):
#     text = re.sub(r"```json|```", "", text)

#     m = re.search(r"\{[\s\S]*\}", text)
#     if not m:
#         return []
    
#     json_str = m.group(0)
    
#     json_str = re.sub(r'"([a-zA-Z_]+):\s*(\[\])"', r'"\1": \2', json_str)

#     json_str = re.sub(r'"([a-zA-Z_]+):\s*([0-9\.]+)', r'"\1": \2', json_str)

#     json_str = re.sub(r'"([a-zA-Z_]+):', r'"\1":', json_str)
    
#     json_str = re.sub(r',\s*}', '}', json_str)
#     json_str = re.sub(r',\s*]', ']', json_str)
    
#     json_str = json_str.replace('\xa0', ' ')

#     try:
#         data = json5.loads(json_str)
#         return data.get("assignments", [])
#     except Exception as e:
#         return []
import re
import json5

def extract_assignments(text):
    text = re.sub(r"```json|```", "", text)

    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return []
    
    json_str = m.group(0)
    
    json_str = re.sub(r'"([a-zA-Z_]+):\s*(\[\])"', r'"\1": \2', json_str)

    json_str = re.sub(r'"([a-zA-Z_]+):\s*([0-9\.]+)', r'"\1": \2', json_str)
    
    json_str = re.sub(r',\s*}', '}', json_str)
    json_str = re.sub(r',\s*]', ']', json_str)
    
    json_str = json_str.replace('\xa0', ' ')

    if not re.search(r']\s*}$', json_str):
        json_str = re.sub(r'}\s*}$', '}\n  ]\n}', json_str)

    try:
        data = json5.loads(json_str)
        return data.get("assignments", [])
    except Exception as e:
        print(f"\n--- JSON PARSE ERROR ---\nError: {e}\nFailed String:\n{json_str}\n------------------------\n")
        return []

def parse_assignment_response(response: str, n_docs: int, topics: dict, min_confidence: float = 0.6):

    assignments_raw = extract_assignments(response)
    if not assignments_raw:
        return []

    valid_ids = set(topics.keys())
    assignments = []

    for item in assignments_raw:

        doc_id = item.get("doc_id")
        topic_id = item.get("topic_id")
        if isinstance(topic_id, list):
            topic_id = topic_id[0] if topic_id else -1
        try:
            topic_id = int(topic_id)
        except:
            topic_id = -1
        label = item.get("topic_label", "").strip()
        confidence = float(item.get("confidence", 0.0))
        keywords = item.get("keywords", [])

        if doc_id is None or doc_id < 1 or doc_id > n_docs:
            continue

        doc_idx = doc_id - 1

        keywords = [
            str(k).strip().lower().replace(" ", "_")
            for k in keywords if str(k).strip()
        ]

        if confidence < min_confidence or topic_id not in valid_ids:
            topic_id = -1
            label = "outlier"

        assignments.append({
            "doc_idx": doc_idx,
            "topic_id": topic_id,
            "topic_label": label,
            "keywords": keywords,
            "confidence": confidence
        })

    return assignments

def assign_topics(texts: list, topics: dict, subject: str) -> pd.DataFrame:
    """Stage 3: Assign topics to all documents."""
    
    checkpoint = load_checkpoint("assignment", subject)
    if checkpoint is not None:
        results = checkpoint["results"]
        processed_idx = checkpoint["processed_idx"]
        print(f"  Resumed: {len(results)} assignments, processed up to idx {processed_idx}")
    else:
        results = []
        processed_idx = 0
    
    topics_str = format_topics_for_prompt(topics)
    assigned_doc_indices = {r["doc_idx"] for r in results}
    
    # Process in batches
    total_docs = len(texts)
    pbar = tqdm(total=total_docs, desc="Assigning topics", initial=processed_idx)
    
    idx = processed_idx
    while idx < total_docs:
        batch_end = min(idx + ASSIGNMENT_BATCH_SIZE, total_docs)
        batch_texts = texts[idx:batch_end]
        batch_size = len(batch_texts)
        
        docs_str = "\n\n".join([f"Document {i+1}:\n{t}" for i, t in enumerate(batch_texts)])
        
        user_prompt = ASSIGNMENT_USER_TEMPLATE.format(
            topics=topics_str,
            documents=docs_str
        )
        
        response = call_llm(ASSIGNMENT_SYSTEM_PROMPT, user_prompt)
        parsed = parse_assignment_response(response, batch_size, topics)
        
        # Map relative doc indices to absolute indices
        for a in parsed:
            abs_idx = idx + a["doc_idx"]

            if abs_idx not in assigned_doc_indices:
                results.append({
                    "doc_idx": abs_idx,
                    "topic_id": a["topic_id"],
                    "topic_label": a["topic_label"],
                    "keywords": a["keywords"],
                    "confidence": a["confidence"]
                })
                assigned_doc_indices.add(abs_idx)
        
        # For docs that didn't get assigned (parsing failure), assign fallback
        for i in range(batch_size):
            abs_idx = idx + i
            if abs_idx not in assigned_doc_indices:
                print(f"[LLM Failure] doc {abs_idx} unresolved → assigned outlier")
                results.append({
                    "doc_idx": abs_idx,
                    "topic_id": -1,
                    "topic_label": "outlier", 
                    "keywords": []
                })
                assigned_doc_indices.add(abs_idx)
        
        idx = batch_end
        pbar.update(batch_size)
        
        if idx % 50 == 0 or idx >= total_docs:
            save_checkpoint({"results": results, "processed_idx": idx}, "assignment", subject)
    
    pbar.close()
    
    # Final save
    save_checkpoint({"results": results, "processed_idx": idx}, "assignment", subject)
    
    # Build DataFrame
    assignment_df = pd.DataFrame(results)
    assignment_df = assignment_df.sort_values("doc_idx").reset_index(drop=True)
    
    print(f"  Assigned {len(assignment_df)} documents to {assignment_df['topic_id'].nunique()} topics")
    return assignment_df


In [ ]:
# Run Topic Assignment
all_assignments = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC ASSIGNMENT: {subject.upper()}")
    print(f"{'='*60}")
    
    texts = all_data[subject]["text"].fillna("").tolist()
    topics = load_checkpoint("refinement", subject)
    if topics is None:
        print(f"  [!] Cannot find refinement checkpoint for {subject}. Skipping...")
        continue

    print(f"\n  Topics for {subject}:")
    for tid, t in sorted(topics.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")
    
    assignment_df = assign_topics(texts, topics, subject)
    all_assignments[subject] = assignment_df
    
    print(f"\n  Topic distribution:")
    dist = assignment_df["topic_label"].value_counts()
    for label, count in dist.head(15).items():
        print(f"    {label}: {count} docs ({count/len(assignment_df)*100:.1f}%)")


TOPIC ASSIGNMENT: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/refinement.pkl

  Topics for cs:
    [1] Graph Neural Networks and Optimization Algorithms: Advanced models for heterogeneous data representation and relational reasoning in GNNs, including optimization algorithms for dynamic problem-solving.
    [2] Adversarial Robustness and Training: Techniques for defending ML systems against input perturbations and adversarial attacks, including optimization of gradient-based adversarial training methods.
    [3] Neural Architecture Pruning and Optimization: Efficient model compression via parameter reduction (pruning) and optimization techniques for balancing computational cost with performance.
    [4] **Cooperative Learning Systems: ** Multi-agent information exchange for unified perception tasks.
    [5] **Knowledge Representation Corpora: ** Large-scale annotated datasets linking concepts to structured knowledge bases and textual definitions.
    [6] **Single-Document S

Assigning topics:   3%|▎         | 5700/165756 [00:00<?, ?it/s]

Assigning topics:   3%|▎         | 5750/165756 [01:02<54:35:47,  1.23s/it]

  Checkpoint saved: ../../../../models/topicGpt/cs/assignment.pkl


Assigning topics:   3%|▎         | 5800/165756 [02:02<53:35:22,  1.21s/it]

...

Assigning topics:   4%|▍         | 6800/165756 [22:54<55:45:46,  1.26s/it]

  Checkpoint saved: ../../../../models/topicGpt/cs/assignment.pkl


Assigning topics:   4%|▍         | 6850/165756 [23:56<53:40:19,  1.22s/it]

  Checkpoint saved: ../../../../models/topicGpt/cs/assignment.pkl


Assigning topics:   4%|▍         | 6880/165756 [24:33<54:30:18,  1.24s/it]

---
## Extract Topic Words for Coherence

Aggregate keywords per topic from assignments, supplemented by TF-IDF from assigned documents.

In [ ]:
def extract_topic_words(assignment_df: pd.DataFrame, texts: list, topics: dict, top_n: int = 10) -> dict:
    """Extract top-N representative words per topic.
    
    Combines LLM-extracted keywords with TF-IDF from assigned documents.
    """
    topic_words = {}  # {topic_id: [word1, word2, ...]}
    
    for topic_id in sorted(topics.keys()):
        # Get docs assigned to this topic
        mask = assignment_df["topic_id"] == topic_id
        assigned_rows = assignment_df[mask]
        
        if len(assigned_rows) == 0:
            topic_words[topic_id] = []
            continue
        
        # Collect LLM-extracted keywords
        llm_keywords = []
        for _, row in assigned_rows.iterrows():
            if isinstance(row.get("keywords"), list):
                llm_keywords.extend(row["keywords"])
        
        # Collect word frequencies from assigned documents
        word_freq = Counter()
        doc_indices = assigned_rows["doc_idx"].tolist()
        for di in doc_indices:
            if di < len(texts):
                words = texts[di].split()
                word_freq.update(words)
        
        # Compute TF-IDF-like score: prioritize words frequent in topic but not in corpus
        total_docs = len(texts)
        topic_doc_count = len(doc_indices)
        
        # Global document frequency
        global_df = Counter()
        for t in texts:
            global_df.update(set(t.split()))
        
        scored_words = {}
        for word, tf in word_freq.items():
            if len(word) < 3:  # skip very short words
                continue
            df = global_df.get(word, 1)
            idf = np.log(total_docs / (1 + df))
            tf_norm = tf / topic_doc_count
            scored_words[word] = tf_norm * idf
        
        # Boost LLM-extracted keywords
        llm_keyword_counts = Counter(llm_keywords)
        for kw, count in llm_keyword_counts.items():
            kw_clean = kw.replace("_", " ")  # undo underscore join for matching
            # Try both with and without underscore
            for variant in [kw, kw_clean]:
                if variant in scored_words:
                    scored_words[variant] *= (1 + 0.5 * count)
                else:
                    # Add LLM keyword even if not in vocab (with moderate score)
                    scored_words[variant] = count * 0.5
        
        # Get top N
        sorted_words = sorted(scored_words.items(), key=lambda x: -x[1])
        topic_words[topic_id] = [w for w, _ in sorted_words[:top_n]]
    
    return topic_words

In [ ]:
# Extract topic words
all_topic_words = {}
# def load_assignments(subject):
#     path = CHECKPOINT_DIR / subject/ "assignment.pkl"
    
#     if not path.exists():
#         raise FileNotFoundError(f"Assignment file not found: {path}")
    
#     with open(path, "rb") as f:
#         return pickle.load(f)

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"EXTRACT TOPIC WORDS: {subject.upper()}")
    print(f"{'='*60}")
    
    texts = all_data[subject]["text"].fillna("").tolist()
    topics = all_refined_topics[subject]
    assignment_df = all_assignments[subject]

    topic_words = extract_topic_words(assignment_df, texts, topics, top_n=TOP_N_WORDS)
    all_topic_words[subject] = topic_words
    
    for tid, words in sorted(topic_words.items()):
        label = topics[tid]["label"]
        n_docs = (assignment_df["topic_id"] == tid).sum()
        print(f"  [{tid}] {label} ({n_docs} docs): {', '.join(words[:8])}")


EXTRACT TOPIC WORDS: CS


NameError: name 'all_refined_topics' is not defined

---
## Evaluation: Coherence, Diversity, Topic Quality

In [ ]:
def compute_coherence_cv(topic_words: dict, texts: list) -> float:
    """Compute C_v coherence using Gensim."""
    # Tokenize texts
    tokenized = [t.split() for t in texts]
    dictionary = Dictionary(tokenized)
    dictionary.filter_extremes(no_below=10, no_above=0.5)
    
    # Prepare topic word lists (filter to words in vocab)
    valid_topics = []
    for tid, words in sorted(topic_words.items()):
        valid_words = [w for w in words if w in dictionary.token2id]
        if len(valid_words) >= 3:  # need at least 3 words for meaningful coherence
            valid_topics.append(valid_words)
    
    if not valid_topics:
        print("  WARNING: No valid topics for coherence computation")
        return 0.0
    
    cm = CoherenceModel(
        topics=valid_topics,
        texts=tokenized,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return cm.get_coherence()


def compute_irbo_diversity(topic_words: dict, top_n: int = 10, p: float = 0.9) -> float:
    """Compute IRBO (Inverted Rank-Biased Overlap) diversity.
    
    IRBO = 1 - avg(RBO) across all topic pairs.
    Higher = more diverse topics.
    """
    word_lists = [words[:top_n] for words in topic_words.values() if len(words) > 0]
    
    if len(word_lists) < 2:
        return 1.0
    
    def rbo(list1, list2, p=0.9):
        """Rank-Biased Overlap between two ranked lists."""
        k = max(len(list1), len(list2))
        if k == 0:
            return 0.0
        
        rbo_val = 0.0
        for d in range(1, k + 1):
            set1 = set(list1[:d])
            set2 = set(list2[:d])
            if len(set1) == 0 and len(set2) == 0:
                continue
            overlap = len(set1 & set2)
            agreement = overlap / d
            rbo_val += (p ** (d - 1)) * agreement
        
        rbo_val *= (1 - p)
        return rbo_val
    
    rbo_scores = []
    for i, j in combinations(range(len(word_lists)), 2):
        rbo_scores.append(rbo(word_lists[i], word_lists[j], p))
    
    avg_rbo = np.mean(rbo_scores)
    return 1.0 - avg_rbo


def compute_topic_quality(coherence: float, diversity: float) -> float:
    """Topic Quality = harmonic mean of coherence and diversity."""
    if coherence + diversity == 0:
        return 0.0
    return 2 * (coherence * diversity) / (coherence + diversity)

In [ ]:
# Compute metrics for all subjects
results = []

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"EVALUATION: {subject.upper()}")
    print(f"{'='*60}")
    
    texts = all_data[subject]["text"].fillna("").tolist()
    topics = all_refined_topics[subject]
    topic_words = all_topic_words[subject]
    
    n_topics = len(topics)
    print(f"  Number of topics: {n_topics}")
    
    # Coherence
    print(f"  Computing C_v coherence...")
    coherence = compute_coherence_cv(topic_words, texts)
    print(f"  Coherence (C_v): {coherence:.4f}")
    
    # Diversity
    diversity = compute_irbo_diversity(topic_words, top_n=TOP_N_WORDS, p=RBO_P)
    print(f"  IRBO Diversity: {diversity:.4f}")
    
    # Topic Quality
    quality = compute_topic_quality(coherence, diversity)
    print(f"  Topic Quality: {quality:.4f}")
    
    results.append({
        "subject": subject,
        "model": "topicGpt",
        "n_topics": n_topics,
        "coherence": coherence,
        "diversity_irbo": diversity,
        "topic_quality": quality
    })

results_df = pd.DataFrame(results)
print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(results_df.to_string(index=False))

---
## Save Results

In [ ]:
# Save coherence results (main summary)
results_df.to_csv(RESULT_DIR / f"coherence_results_{VERSION}.csv", index=False)
print(f"Saved: {RESULT_DIR / f'coherence_results_{VERSION}.csv'}")

# Save per-subject detailed results
for subject in LIST_SUBJECT:
    out_dir = RESULT_DIR / subject
    topics = all_refined_topics[subject]
    assignment_df = all_assignments[subject]
    topic_words = all_topic_words[subject]
    
    # 1. Topics CSV
    topics_rows = []
    for tid, t in sorted(topics.items()):
        n_docs = (assignment_df["topic_id"] == tid).sum()
        words = topic_words.get(tid, [])
        topics_rows.append({
            "topic_id": tid,
            "label": t["label"],
            "description": t["description"],
            "n_docs": n_docs,
            "top_words": ", ".join(words)
        })
    pd.DataFrame(topics_rows).to_csv(out_dir / "topics.csv", index=False)
    
    # 2. Assignments CSV
    assign_out = assignment_df[["doc_idx", "topic_id", "topic_label"]].copy()
    assign_out["keywords"] = assignment_df["keywords"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else ""
    )
    assign_out.to_csv(out_dir / "assignments.csv", index=False)
    
    # 3. Topic Words CSV
    tw_rows = []
    for tid, words in sorted(topic_words.items()):
        label = topics[tid]["label"]
        for rank, w in enumerate(words, 1):
            tw_rows.append({
                "topic_id": tid,
                "topic_label": label,
                "rank": rank,
                "word": w
            })
    pd.DataFrame(tw_rows).to_csv(out_dir / "topic_words.csv", index=False)
    
    print(f"Saved {subject}: topics.csv, assignments.csv, topic_words.csv")

print(f"\nAll results saved to: {RESULT_DIR.resolve()}")

In [ ]:
# Final comparison display
print("\n" + "="*70)
print("FINAL RESULTS: TopicGPT Coherence Scores")
print("="*70)

for _, row in results_df.iterrows():
    print(f"\n  {row['subject'].upper()}:")
    print(f"    Topics:     {row['n_topics']}")
    print(f"    Coherence:  {row['coherence']:.4f}")
    print(f"    Diversity:  {row['diversity_irbo']:.4f}")
    print(f"    Quality:    {row['topic_quality']:.4f}")

print(f"\n{'─'*70}")
print(f"  Average Coherence:  {results_df['coherence'].mean():.4f}")
print(f"  Average Diversity:  {results_df['diversity_irbo'].mean():.4f}")
print(f"  Average Quality:    {results_df['topic_quality'].mean():.4f}")